In [1]:
# Importing Modules
import torch
import numpy as np
import pandas as pd
from src.dataloader import loadData
from src.model import GNNModel
from src.train_test import TestGNN
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from torch.optim import Adam
from torch.nn.functional import mse_loss

In [2]:
######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/train/RT.csv")
rt_val_data = pd.read_csv("data/val/RT.csv")
rt_test_data = pd.read_csv("data/test/RT.csv")

In [3]:
# Function to fix GNN layers
def freeze_gnn_layers(model):

	# Fixed layers
	for param in model.input_proj.parameters():
		param.requires_grad = False
	for param in model.convs.parameters():
		param.requires_grad = False
	for param in model.batch_norms.parameters():
		param.requires_grad = False

	# Trainable layers
	for param in model.post_mp.parameters():
		param.requires_grad = True

In [4]:
# Run GNN analysis
def RunGNN(train_data, val_data, test_data, dataName, modelName, params):

	# Training data and target labels
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	y_train = train_data["target"]
	y_train = y_train.to_numpy()

	# Validation data and target labels
	smiles_X_val = val_data["smiles"]
	X_val = smiles_X_val.to_numpy()
	y_val = val_data["target"]
	y_val = y_val.to_numpy()

	# Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

	# Params
	h_dim, b_size, lr, d_out, wd, layers = params

	# Loader
	train_loader = loadData(X_train, y_train, batch_size=b_size, shuf=True) # Usually good to shuffle train
	val_loader = loadData(X_val, y_val, batch_size=b_size, shuf=False)
	test_loader = loadData(X_test, y_test, batch_size=b_size, shuf=False)

	# Device
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

	# Model
	model = torch.load(f"results/{modelName}_METLIN.pth", weights_only=False, map_location=device)

	# Freezing model wts
	freeze_gnn_layers(model) 

	# Upload to device
	model.to(device)

	# Setup optimizer as per updated model
	trainable_params = [p for p in model.parameters() if p.requires_grad]
	optimizer = torch.optim.Adam(trainable_params, lr=lr, weight_decay=wd)


    # Model Training
	patience = 10
	best_val_loss = float('inf')
	epochs_without_improvement = 0
	best_model_weights = model.state_dict().copy()

	for e in range(100):
		model.train()
		train_loss = 0
		for data in train_loader: # Fixed: matched variable name from above
			data = data.to(device)
			optimizer.zero_grad()
			out, _ = model(data)
			loss = mse_loss(out.view(-1), data.y.view(-1)) 
			loss.backward()
			optimizer.step()
			train_loss += loss.item()

		model.eval()
		val_loss = 0
		with torch.no_grad():
			for data in val_loader:
				data = data.to(device)
				out, _ = model(data)
				loss = mse_loss(out.view(-1), data.y.view(-1))
				val_loss += loss.item()

		avg_train_loss = train_loss / len(train_loader)
		avg_val_loss = val_loss / len(val_loader)

		if avg_val_loss < best_val_loss:
			best_val_loss = avg_val_loss
			epochs_without_improvement = 0
			best_model_weights = model.state_dict().copy()

		else:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			model.load_state_dict(best_model_weights)
			break

    # Model Testing
	y_test_out, y_pred_out = TestGNN(model, test_loader)
	return root_mean_squared_error(y_test_out, y_pred_out)

In [5]:
# Data dict
datasets = {"RT":{"train":rt_train_data, "val":rt_val_data, "test":rt_test_data}}

# Models
models = [
    "GCN", "GAT", "GIN", "GraphSAGE"]

In [6]:
temp_out = []

for modelName in models:
	for dataName, data in datasets.items():
		# Load best GNN hyperparameters
		temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_METLIN.csv")
		temp = temp_df[temp_df["Model"] == modelName]
		
		gnn_params = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay", "layers"]].to_dict('records')[0].values()

		rmse_list = []
		num_runs = 3

		for i in range(num_runs):
			rmse = RunGNN(data["train"], data["val"], data["test"], dataName, modelName, gnn_params)
			rmse_list.append(rmse)

		rmse_mean, rmse_std = np.mean(rmse_list), np.std(rmse_list)
		rmse_str = f"{rmse_mean:.3f} ({rmse_std:.3f})"

		best_entry = pd.DataFrame({
			"GNN": [modelName], 
			"Dataset": [dataName], 
			"RMSE": [rmse_str],
			})
		
		temp_out.append(best_entry)

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNNTransfer_Analysis.csv", index=False)
GNN_out

,GNN,Dataset,RMSE
0,GCN,RT,122.069 (1.699)
1,GAT,RT,115.132 (4.036)
2,GIN,RT,80.340 (1.006)
3,GraphSAGE,RT,105.146 (0.338)
